In [ ]:
# Colab reproducibility — uncomment to clone the repo and enter this folder:
# !git clone https://github.com/JuliettKhar/mlops-zoomcamp-exercises.git
# %cd mlops-zoomcamp-exercises/03-ml-pipelines

In [ ]:
# Colab setup — uncomment to install packages not preinstalled in Colab:
# !pip install mlflow

In [ ]:
# Q1
# Prefect

In [ ]:
# Q2
# Version:              3.7.2

In [ ]:
# Q3
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
import sklearn

filename_green = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-01.parquet'
filename_yellow = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet'

df = pd.read_parquet(filename_yellow)
print("Q3 records:", len(df))



Q3 records: 3724889


In [ ]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    # df.tpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)
    # df.tpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)

    # df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df["duration"] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.apply( lambda td: td.total_seconds() / 60)

    # df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [26]:

# Q4
df_2 = read_dataframe(filename_yellow)
print("Q4 prepared records:", len(df_2))

Q4 prepared records: 3571924


In [27]:
df_2.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,duration
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,...,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00,5.550000
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,...,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75,5.716667
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,...,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75,8.883333
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,...,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75,42.800000
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,...,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75,13.500000


In [19]:
categorical = ['PULocationID', 'DOLocationID']
train_dict = df_2[categorical].to_dict(orient='records')

In [ ]:
dv = DictVectorizer()
x_train = dict.fit_transform(train_dict)
y_train = df_2.duration.values

In [29]:
lr = LinearRegression()
lr.fit(x_train, y_train)
print("Q5 intercept:", lr.intercept_)
# y_pred = lr.predict(x_train)
# rmse = root_mean_squared_error(y_train, y_pred)

Q5 intercept: 24.334958011863506


In [31]:
import mlflow

with mlflow.start_run():
    mlflow.sklearn.log_model(
        sk_model=lr,
        name="model"
    )

2026/05/31 11:54:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
# Q6
# model_size_bytes: 4540